# 1.1 Build the Cohorts

**Class 1:** pull the data for our study from the All of Us dataset (CDR) with SQL.

**Study question:** how do people **with** and **without diabetes** differ in demographics, PTSD care, and cardiometabolic measurements?

**Study population** (both groups):

- Age 25–50, alive
- Race: White, Black, or Asian
- Ethnicity: Hispanic or not Hispanic
- Gender: Male or Female
- **Study group:** has a diabetes diagnosis. **Control group:** doesn't.

**We pull three tables:**

| Table | Variables | Saved as (study / control) |
|---|---|---|
| Demographics | date of birth, gender, race, ethnicity | `person_study.csv` / `person_control.csv` |
| Survey | two PTSD questions: still seeing a doctor? currently treated? | `survey_study.csv` / `survey_control.csv` |
| Measurements | BMI, systolic blood pressure, LDL cholesterol, HbA1c | `measurement_study.csv` / `measurement_control.csv` |

Each group is saved to **its own file**. In notebook **2.0** we flag which group each person belongs to and bring the groups together.

> 💡 **Run the queries once.** They cost money and take a few minutes. After that, later notebooks read the saved CSVs.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

## About these queries

The SQL below was **generated by the classic Workbench's Cohort Builder** for our two cohorts. To run it in Workbench 2.0 we changed only two things:

1. **The dataset path.** Every table is written as `` `{CURRENT_DATASET}.table` ``, so it points at the dataset set in `aou_setup.R`:

   | | |
   |---|---|
   | Classic Workbench ❌ | `` FROM `fc-aou-cdr-prod-ct.C2024Q3R8.person` `` |
   | Workbench 2.0 ✅ | `` FROM `{CURRENT_DATASET}.person` `` |

2. **One word for the two groups.** The study and control queries were identical except that the study group **includes** people with diabetes (`IN`) and the control group **excludes** them (`NOT IN`). We replaced that word with `{diabetes_filter}`, so each query is written once and run twice: `diabetes_filter = "IN"` for the study group and `"NOT IN"` for the control group.

`str_glue()` fills in `{CURRENT_DATASET}` and `{diabetes_filter}` before the query runs.

## Check that the tables exist

Besides the standard OMOP tables (`person`, `measurement`…), the queries use Cohort Builder tables (`cb_search_person`, `cb_criteria`, `cb_search_all_events`) and `ds_survey`. This lists which ones your dataset has. **All of them should appear**; if some are missing, use the Data Explorer route at the end of this notebook instead.

In [ ]:
needed <- c("person", "concept", "death", "measurement", "ds_survey",
            "cb_search_person", "cb_criteria", "cb_search_all_events")

found <- run_query(str_glue(
  "SELECT table_name FROM `{CURRENT_DATASET}`.INFORMATION_SCHEMA.TABLES"))$table_name

tibble(table = needed, found = needed %in% found)

## Query 1: Demographics

The queries are written as R **raw strings**, `r"---( … )---"`, so the SQL can be pasted in exactly as the Cohort Builder wrote it.

In [ ]:
person_sql <- r"---(
      SELECT
        person.person_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        p_race_concept.concept_name as race,
        p_ethnicity_concept.concept_name as ethnicity,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `{CURRENT_DATASET}.person` person 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `{CURRENT_DATASET}.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{CURRENT_DATASET}.cb_search_person` p 
            WHERE
                DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                AND NOT EXISTS (      SELECT
                    'x'      
                FROM
                    `{CURRENT_DATASET}.death` d      
                WHERE
                    d.person_id = p.person_id ) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{CURRENT_DATASET}.person` p 
            WHERE
                ethnicity_concept_id IN (38003564, 38003563) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{CURRENT_DATASET}.person` p 
            WHERE
                gender_concept_id IN (45878463, 45880669) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{CURRENT_DATASET}.person` p 
            WHERE
                race_concept_id IN (8527, 8516, 8515) ) 
            AND cb_search_person.person_id {diabetes_filter} (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `{CURRENT_DATASET}.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `{CURRENT_DATASET}.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `{CURRENT_DATASET}.cb_criteria` cr       
                        WHERE
                            concept_id IN (201820)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )
)---"

person_study        <- run_query(str_glue(person_sql, diabetes_filter = "IN"))      # has diabetes
person_control      <- run_query(str_glue(person_sql, diabetes_filter = "NOT IN"))  # no diabetes

glimpse(person_study)

In [ ]:
save_data(person_study,   "person_study.csv")
save_data(person_control, "person_control.csv")

## Query 2: Survey answers

The two PTSD questions (concepts 43528846 and 43530361), from the `ds_survey` table.

In [ ]:
survey_sql <- r"---(
    SELECT
        answer.person_id,
        answer.question,
        answer.answer  
    FROM
        `{CURRENT_DATASET}.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (43528846, 43530361)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `{CURRENT_DATASET}.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.cb_search_person` p 
                WHERE
                    DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                    AND NOT EXISTS (      SELECT
                        'x'      
                    FROM
                        `{CURRENT_DATASET}.death` d      
                    WHERE
                        d.person_id = p.person_id ) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    ethnicity_concept_id IN (38003564, 38003563) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    gender_concept_id IN (45878463, 45880669) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    race_concept_id IN (8527, 8516, 8515) ) 
                AND cb_search_person.person_id {diabetes_filter} (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `{CURRENT_DATASET}.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `{CURRENT_DATASET}.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `{CURRENT_DATASET}.cb_criteria` cr       
                            WHERE
                                concept_id IN (201820)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 )) criteria ) )
        )
)---"

survey_study        <- run_query(str_glue(survey_sql, diabetes_filter = "IN"))      # has diabetes
survey_control      <- run_query(str_glue(survey_sql, diabetes_filter = "NOT IN"))  # no diabetes

glimpse(survey_study)

Check the question and answer wording. Notebook 2.0 matches on it:

In [ ]:
survey_study %>% distinct(question, answer)

In [ ]:
save_data(survey_study,   "survey_study.csv")
save_data(survey_control, "survey_control.csv")

## Query 3: Measurements

BMI, systolic blood pressure, LDL cholesterol, and HbA1c. This is the biggest query: **millions of rows** (one per measurement). Notebook 2.0 keeps the most recent value per person.

In [ ]:
measurement_sql <- r"---(
      SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        measurement.measurement_datetime,
        measurement.value_as_number 
    FROM
        ( SELECT
            * 
        FROM
            `{CURRENT_DATASET}.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `{CURRENT_DATASET}.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `{CURRENT_DATASET}.cb_criteria` cr       
                    WHERE
                        concept_id IN (3004249, 3004410, 3038553, 40795800, 4152194)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `{CURRENT_DATASET}.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.cb_search_person` p 
                    WHERE
                        DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                        AND NOT EXISTS (      SELECT
                            'x'      
                        FROM
                            `{CURRENT_DATASET}.death` d      
                        WHERE
                            d.person_id = p.person_id ) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        ethnicity_concept_id IN (38003564, 38003563) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        gender_concept_id IN (45878463, 45880669) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        race_concept_id IN (8527, 8516, 8515) ) 
                    AND cb_search_person.person_id {diabetes_filter} (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `{CURRENT_DATASET}.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `{CURRENT_DATASET}.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `{CURRENT_DATASET}.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
            )) measurement 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id
)---"

measurement_study   <- run_query(str_glue(measurement_sql, diabetes_filter = "IN"))      # has diabetes
measurement_control <- run_query(str_glue(measurement_sql, diabetes_filter = "NOT IN"))  # no diabetes

glimpse(measurement_study)

In [ ]:
save_data(measurement_study,   "measurement_study.csv")
save_data(measurement_control, "measurement_control.csv")

## Check what you saved

These files are now in your bucket's `data/` folder. You won't need to run the queries again.

In [ ]:
list.files(DATA_DIR)

---

## Alternative: use a Data Explorer export instead

You can build the same two groups with **Data Explorer** instead of running SQL. **Step-by-step instructions for both cohorts are in the notebooks README:** [notebooks/README.md](https://github.com/CWML/All-of-Us-Training-Resources/tree/main/notebooks#build-the-cohorts-in-data-explorer). In short:

1. Add **two cohorts**, using the criteria at the top of this notebook: one **with** diabetes (Condition: *Diabetes mellitus*) and one **without** (exclude it)
2. For each cohort, **Save data snapshot** with *Demographics*, the two PTSD survey questions, and the four measurements, as an **R notebook**
3. Run each exported notebook once. It saves CSV files to your bucket under `bq_exports/`
4. Then use the code below to read them and save them with the **same file names and columns** as above, so notebooks 2.0 to 3.4 work unchanged

The exported **person** table uses different column names (e.g. `T_DISP_gender`). This renames them:

In [ ]:
# Find the exported files (they're in folders named by table and date)
list.files(file.path(BUCKET_DIR, "bq_exports"), recursive = TRUE, pattern = "\\.csv$")

In [ ]:
# Read every CSV in one export folder into a single data frame
read_export <- function(folder) {
  list.files(folder, pattern = "\\.csv$", full.names = TRUE) %>%
    map_dfr(read_csv, show_col_types = FALSE)
}

# EDIT these paths: one person export folder per cohort
person_study   <- read_export(file.path(BUCKET_DIR, "bq_exports", "EDIT/study/person/folder"))
person_control <- read_export(file.path(BUCKET_DIR, "bq_exports", "EDIT/control/person/folder"))

# Rename the export's columns to match the SQL version
tidy_person <- function(df) {
  df %>% transmute(person_id,
                   date_of_birth = birth_datetime,
                   gender    = T_DISP_gender,
                   race      = T_DISP_race,
                   ethnicity = T_DISP_ethnicity)
}
person_study   <- tidy_person(person_study)
person_control <- tidy_person(person_control)

# save_data(person_study,   "person_study.csv")     # uncomment to save
# save_data(person_control, "person_control.csv")

Do the same for the **survey** and **measurement** exports: read each cohort, rename columns to match the SQL versions above, and save as `survey_study.csv`, `survey_control.csv`, `measurement_study.csv`, and `measurement_control.csv` (`person_id`, `question`, `answer`; and `person_id`, `measurement_concept_id`, `standard_concept_name`, `measurement_datetime`, `value_as_number`). Use `names()` to see what the export calls them.